In [ ]:

import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from gymnasium import spaces
from typing import Optional  # Python 3.9

import myosuite  



class CFG:
    ENV_ID = "motorFingerPoseFixed-v0"   

    # real data frame index window (딱밤 동작이 들어있는 구간)
    WINDOW_START = 800
    WINDOW_END   = 950    

    EPISODE_LEN  = 150    

    REWARD_SCALE = 0.05   
    W_SHAPE      = 0.10   
    W_FINAL      = 0.50   

    TOTAL_UPDATES     = 10000    
    STEPS_PER_UPDATE  = 1024     
    GAMMA             = 0.99
    LAMBDA            = 0.95
    CLIP_EPS          = 0.2
    LR                = 1e-5
    BATCH_SIZE        = 256
    PPO_EPOCHS        = 5

    BC_KP            = 1.0       
    BC_LAMBDA        = 0.1       
    BC_WARM_UPDATES  = 500       

    SEED   = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)



def load_middle_finger_targets(npy_path: str, cfg: CFG) -> np.ndarray:
    """
    myohand_joint_angles_23dof_stable_signed.npy 에서
    중지 MCP/PIP/DIP flexion (qpos[11], qpos[13], qpos[14])만 뽑아서 (T_full, 3) 반환,
    그리고 [WINDOW_START:WINDOW_END] 구간만 잘라서 사용.
    """
    qpos_seq = np.load(npy_path)   # (T_full, 23) 예상
    print("[INFO] Loaded NPY:", npy_path, "shape =", qpos_seq.shape)

    idx = [11, 13, 14]
    middle_full = qpos_seq[:, idx]   # (T_full, 3)

    print("[INFO] Full middle finger angles shape:", middle_full.shape)
    print("[DEBUG] Full first frame (rad):", middle_full[0])
    print("[DEBUG] Full first frame (deg):", np.rad2deg(middle_full[0]))

    # --- 800~950 구간만 사용 ---
    start = cfg.WINDOW_START
    end   = cfg.WINDOW_END
    middle_win = middle_full[start:end]   # (T_win, 3)

    print(f"[INFO] Using window [{start}:{end}) → shape:", middle_win.shape)
    print("[DEBUG] Window first frame (deg):", np.rad2deg(middle_win[0]))
    print("[DEBUG] Window last  frame (deg):", np.rad2deg(middle_win[-1]))

    return middle_win.astype(np.float32)


npy_path = r"C:\Users\Donggyu\Downloads\myosuite\myohand_joint_angles_23dof_stable_signed.npy"
target_angles = load_middle_finger_targets(npy_path, cfg)

cfg.EPISODE_LEN = target_angles.shape[0]
print("[INFO] cfg.EPISODE_LEN set to T_win =", cfg.EPISODE_LEN)


# %% ==============================
# 3. motorFinger wrapper env (패턴 reward + final-state bonus)
# ==============================
class MotorFingerTrajEnv(gym.Env):
    
    metadata = {"render_modes": []}

    def __init__(self, target_angles: np.ndarray, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.base_env = gym.make(cfg.ENV_ID)
        self.sim = self.base_env.unwrapped.sim

        self.target_angles = target_angles       # (T_win, 3)
        self.T = target_angles.shape[0]          # T_win

        mj_model = self.sim.model
        j_mcp = mj_model.joint_name2id("IFmcp")
        j_pip = mj_model.joint_name2id("IFpip")
        j_dip = mj_model.joint_name2id("IFdip")
        j_ids = np.array([j_mcp, j_pip, j_dip], dtype=int)
        self.qpos_adr = mj_model.jnt_qposadr[j_ids]   # (3,)

        low = -np.ones(7, dtype=np.float32) * 10.0
        high = np.ones(7, dtype=np.float32) * 10.0
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        self.action_space = self.base_env.action_space

        self.t = 0  # time index

        angles = self.target_angles  
        split = int(0.5 * self.T)    

        self.flex_ref = angles[:split].mean(axis=0)   # (3,)
        self.ext_ref = angles[split:].mean(axis=0)    # (3,)

        print(f"[INFO] MotorFingerTrajEnv: obs_dim={self.observation_space.shape[0]}, "
              f"act_dim={self.action_space.shape[0]}, T_win={self.T}")
        print("[INFO] flex_ref (deg):", np.rad2deg(self.flex_ref))
        print("[INFO] ext_ref  (deg):", np.rad2deg(self.ext_ref))

    def _get_q(self) -> np.ndarray:
        qpos = self.sim.data.qpos
        return qpos[self.qpos_adr].copy()

    def _build_obs(self) -> np.ndarray:
        q = self._get_q()
        idx = min(self.t, self.T - 1)
        target = self.target_angles[idx]
        frac = np.array([self.t / float(self.cfg.EPISODE_LEN)], dtype=np.float32)
        obs = np.concatenate([q.astype(np.float32), target.astype(np.float32), frac], axis=0)
        return obs

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.base_env.reset(seed=seed)
        self.t = 0

        qpos = self.sim.data.qpos.copy()
        qpos[self.qpos_adr] = self.target_angles[0]
        self.sim.data.qpos[:] = qpos
        self.sim.forward()

        obs = self._build_obs()
        info = {}
        return obs, info

    def step(self, action):
        # base env step
        _, _, terminated_base, truncated_base, info = self.base_env.step(action)

        # 현재 q / target
        q = self._get_q()                         # (3,)
        idx = min(self.t, self.T - 1)
        target = self.target_angles[idx]          # (3,)
        err = q - target                          # (3,)

  
        r_track = - self.cfg.REWARD_SCALE * float(np.sum(err ** 2))

        
        phi = self.t / float(self.cfg.EPISODE_LEN)   # [0, 1]
        if phi < 0.5:
            ref = self.flex_ref   
        else:
            ref = self.ext_ref   

        shape_err = q - ref
        r_shape = - self.cfg.W_SHAPE * float(np.sum(shape_err ** 2))

        if self.t == (self.cfg.EPISODE_LEN - 1):
            final_err = q - self.ext_ref
            r_final = - self.cfg.W_FINAL * float(np.sum(final_err ** 2))
        else:
            r_final = 0.0

        reward = r_track + r_shape + r_final

        
        self.t += 1
        terminated = self.t >= self.cfg.EPISODE_LEN
        truncated = False

        obs = self._build_obs()
        done = (terminated or terminated_base or truncated or truncated_base)
        return obs, reward, done, truncated, info

    def render(self):
        return self.base_env.render()

    def close(self):
        self.base_env.close()


env = MotorFingerTrajEnv(target_angles, cfg)



class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        hidden = 128

        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, action_dim),
        )
        # log_std는 learnable parameter
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        raise NotImplementedError

    def act(self, state):
        
        if not isinstance(state, torch.Tensor):
            state_t = torch.from_numpy(state).float().unsqueeze(0)
        else:
            state_t = state.unsqueeze(0)

        mu = self.actor(state_t)             # (1, A)
        std = torch.exp(self.log_std)        # (A,)
        dist = torch.distributions.Normal(mu, std)
        action = dist.sample()               # (1, A)
        log_prob = dist.log_prob(action).sum(dim=-1)  # (1,)
        value = self.critic(state_t).squeeze(-1)      # (1,)

        return action.detach().cpu().numpy()[0], log_prob.item(), value.item()

    def evaluate_actions(self, states, actions):
       
        mu = self.actor(states)              # (N, A)
        std = torch.exp(self.log_std)        # (A,)
        dist = torch.distributions.Normal(mu, std)

        log_probs = dist.log_prob(actions).sum(dim=-1)  # (N,)
        entropy = dist.entropy().sum(dim=-1)            # (N,)
        values = self.critic(states).squeeze(-1)        # (N,)

        return log_probs, entropy, values



def compute_gae(rewards, values, dones, gamma, lam):
    """
    rewards, values, dones: (T,)
    """
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(T)):
        next_value = values[t + 1] if t + 1 < T else 0.0
        next_non_terminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        last_gae = delta + gamma * lam * next_non_terminal * last_gae
        adv[t] = last_gae
    returns = adv + values
    return adv, returns


def ppo_train(env: MotorFingerTrajEnv, cfg: CFG) -> ActorCritic:
    device = torch.device(cfg.DEVICE)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    ac = ActorCritic(state_dim, action_dim).to(device)
    optimizer = torch.optim.Adam(ac.parameters(), lr=cfg.LR)

    global_step = 0

    for update in range(1, cfg.TOTAL_UPDATES + 1):
        states = []
        actions = []
        rewards = []
        dones = []
        old_log_probs = []
        values = []
        ep_returns = []

        steps_collected = 0

        
        while steps_collected < cfg.STEPS_PER_UPDATE:
            state, _ = env.reset()
            done = False
            t = 0
            ep_ret = 0.0

            while (not done) and (t < cfg.EPISODE_LEN) and (steps_collected < cfg.STEPS_PER_UPDATE):
                action, logp, value = ac.act(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                states.append(state)
                actions.append(action)
                rewards.append(reward)
                dones.append(done)
                old_log_probs.append(logp)
                values.append(value)

                ep_ret += reward
                state = next_state
                t += 1
                steps_collected += 1
                global_step += 1

            ep_returns.append(ep_ret)

    
        states = np.array(states, dtype=np.float32)
        actions = np.array(actions, dtype=np.float32)
        rewards = np.array(rewards, dtype=np.float32)
        dones = np.array(dones, dtype=np.bool_)
        values = np.array(values, dtype=np.float32)

        
        advantages, returns = compute_gae(rewards, values, dones,
                                          gamma=cfg.GAMMA, lam=cfg.LAMBDA)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        states_t = torch.from_numpy(states).float().to(device)
        actions_t = torch.from_numpy(actions).float().to(device)
        old_log_probs_t = torch.from_numpy(np.array(old_log_probs, dtype=np.float32)).to(device)
        returns_t = torch.from_numpy(returns).float().to(device)
        adv_t = torch.from_numpy(advantages).float().to(device)

    
        dataset_size = states_t.shape[0]
        idxs = np.arange(dataset_size)
        last_loss = 0.0

        for epoch in range(cfg.PPO_EPOCHS):
            np.random.shuffle(idxs)
            for start in range(0, dataset_size, cfg.BATCH_SIZE):
                end = start + cfg.BATCH_SIZE
                batch_idx = idxs[start:end]

                b_states = states_t[batch_idx]
                b_actions = actions_t[batch_idx]
                b_old_logp = old_log_probs_t[batch_idx]
                b_returns = returns_t[batch_idx]
                b_adv = adv_t[batch_idx]

                new_logp, entropy, values_pred = ac.evaluate_actions(b_states, b_actions)
                ratio = torch.exp(new_logp - b_old_logp)

                surr1 = ratio * b_adv
                surr2 = torch.clamp(ratio,
                                    1.0 - cfg.CLIP_EPS,
                                    1.0 + cfg.CLIP_EPS) * b_adv

                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = nn.MSELoss()(values_pred, b_returns)
                entropy_loss = -entropy.mean()


                bc_loss = torch.tensor(0.0, device=device)
                if update <= cfg.BC_WARM_UPDATES and cfg.BC_LAMBDA > 0.0:
                    q_t   = b_states[:, 0:3]
                    tgt_t = b_states[:, 3:6]
                    err_t = tgt_t - q_t       # (N, 3)

                    expert_act = torch.zeros_like(ac.actor(b_states))  # (N, action_dim)
                    expert_act[:, 0:3] = cfg.BC_KP * err_t             # 앞 3개만 사용 (heuristic)

                    mu_pred = ac.actor(b_states)
                    bc_loss = ((mu_pred - expert_act) ** 2).mean() * cfg.BC_LAMBDA

                loss = actor_loss + 0.5 * critic_loss + 0.001 * entropy_loss + bc_loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                last_loss = loss.item()

        avg_return = float(np.mean(ep_returns))
        if (update % 10) == 0 or update == 1:
            print(f"[Update {update:4d}/{cfg.TOTAL_UPDATES}] "
                  f"Steps: {global_step:7d}  AvgReturn: {avg_return:7.3f}  LastLoss: {last_loss:7.4f}")

    print("=== Training finished ===")

    # policy 저장
    save_path = "ppo_motorfinger_actor_window800_950_bcWarm.pth"
    actor_state = {
        "actor": ac.actor.state_dict(),
        "log_std": ac.log_std.detach().cpu(),
    }
    torch.save(actor_state, save_path)
    print(f"[INFO] Saved actor (policy) weights to {save_path}")

    return ac



def eval_policy(env: MotorFingerTrajEnv,
                policy: Optional[ActorCritic],
                cfg: CFG,
                episodes: int = 5):
    def run_episode(use_policy: bool):
        state, _ = env.reset()
        done = False
        ep_ret = 0.0
        traj_err = []

        t = 0
        while not done and t < cfg.EPISODE_LEN:
            if use_policy and policy is not None:
                action, _, _ = policy.act(state)
            else:
                low = env.action_space.low
                high = env.action_space.high
                action = np.random.uniform(low, high)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_ret += reward

            q = env._get_q()
            idx = min(env.t, env.T - 1)
            target = env.target_angles[idx]
            traj_err.append(np.linalg.norm(q - target))

            state = next_state
            t += 1

        return ep_ret, float(np.mean(traj_err))

    print("===== Eval random policy =====")
    rets, errs = [], []
    for _ in range(episodes):
        r, e = run_episode(use_policy=False)
        rets.append(r)
        errs.append(e)
    print(f"[Random]   mean return={np.mean(rets):.3f}, mean tracking error={np.mean(errs):.4f}")

    if policy is not None:
        print("===== Eval trained policy =====")
        rets, errs = [], []
        for _ in range(episodes):
            r, e = run_episode(use_policy=True)
            rets.append(r)
            errs.append(e)
        print(f"[Trained]  mean return={np.mean(rets):.3f}, mean tracking error={np.mean(errs):.4f}")



print("\n===== PPO Training (motorFingerPoseFixed, window 800~950 + BC-warm + init-from-800) =====")
policy = ppo_train(env, cfg)

print("\n===== Policy Evaluation =====")
eval_policy(env, policy, cfg, episodes=5)

env.close()
